# Notebook 06: CDR Constraint Lambda Sweep (Experiment 6)

**Strategy:** Delta Residue + Full Wildtype (Exp 3 -- best performing strategy)  
**Input:** `concat(delta_residue[mut_pos], mean_pool(wt_sequence))`  
**Dims:** ESM-2 = 3840, AbLang2 = 1440  
**Lambda sweep:** [0, 0.1, 0.5, 1.0] for both ESM-2 and AbLang2  
**Total runs:** 8 (4 lambdas x 2 models)

The CDR constraint encodes the biological prior that CDR mutations should have
higher predicted effect magnitude than framework mutations:

```
constraint_loss = mean(ReLU(|FR_predicted| - |CDR_predicted|))
total_loss = task_loss + lambda * constraint_loss
```

Lambda=0 must reproduce the Exp 3 unconstrained baseline exactly (sanity check).
All runs logged to W&B.

## Setup

In [ ]:
import subprocess, os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/Aaron1776/antibody-property-prediction.git'
    REPO_DIR = '/content/antibody-property-prediction'
    BRANCH   = 'implementation'

    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
else:
    REPO_DIR = str(Path('..').resolve())
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

print(f"Environment: {'Colab' if IN_COLAB else 'local'}")
print(f"Repo: {REPO_DIR}")

Detects whether running on Colab or locally. On Colab, mounts Drive and clones
(or pulls) the repo. Locally, resolves repo root from the notebook's location.

Expected output: `Environment: local` (or `Colab`) and the resolved repo path.

In [ ]:
from src.config import DRIVE_ROOT, EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR

for d in [EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive root:      {DRIVE_ROOT}")
print(f"Embedding dir:   {EMBEDDING_DIR}")
print(f"Checkpoint dir:  {CHECKPOINT_DIR}")
print("Paths set.")

`src/config.py` resolves `DRIVE_ROOT` automatically across Colab and local.
No per-collaborator edits needed. Checkpoints are saved to Drive so training
can be resumed if Colab disconnects.

In [ ]:
if IN_COLAB:
    subprocess.run(['apt-get', 'install', '-y', 'hmmer'], check=True)
    subprocess.run(['pip', 'install', '-q', '--upgrade', 'ipython'], check=True)
    subprocess.run(['pip', 'install', '-q', 'fair-esm', 'ablang2', 'anarci', 'wandb',
                    'scikit-learn'], check=True)
else:
    print("Local run -- installation skipped.")

In [ ]:
%load_ext autoreload
%autoreload 2

if IN_COLAB:
    subprocess.run(
        ['find', REPO_DIR, '-type', 'd', '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
        capture_output=True,
    )

print("Autoreload enabled.")

In [ ]:
import torch
from src.config import DEVICE

print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif DEVICE == 'mps':
    print("Apple MPS -- Apple Silicon unified memory")

Device check. Training runs on GPU (Colab) or MPS (local Apple Silicon).
Expected: `cuda` on Colab T4/A100, `mps` on Mac.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import wandb
from torch.utils.data import DataLoader, Subset

from src.config import DATA_DIR, EMBEDDING_DIR, DEVICE
from src.data.abagym import load_abagym_antibody
from src.data.datasets import AbAgymDataset, EmbeddingStrategy
from src.data.splits import make_stratified_splits
from src.models.mlp import MLP
from src.training.trainer import TrainConfig, train_abagym, evaluate_abagym

print("Imports OK.")

## Data Loading and Splits

In [ ]:
df = load_abagym_antibody(DATA_DIR)

train_idx, val_idx, test_idx = make_stratified_splits(
    df, val_frac=0.1, test_frac=0.1, random_state=42
)

print(f"Total:  {len(df)}")
print(f"Train:  {len(train_idx)} ({100*len(train_idx)/len(df):.1f}%)")
print(f"Val:    {len(val_idx)} ({100*len(val_idx)/len(df):.1f}%)")
print(f"Test:   {len(test_idx)} ({100*len(test_idx)/len(df):.1f}%)")

Same stratified 80/10/10 split as NB05 (random_state=42). Expected: 4256/531/531.

Record confirmed output here after running.

## Sanity Check: Dataset Dims

In [ ]:
# Verify input dims for DELTA_RESIDUE_PLUS_WILD (Exp 3 strategy)
for model_name in ('esm2', 'ablang2'):
    ds = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_RESIDUE_PLUS_WILD,
        model_name=model_name,
    )
    x0, y0, meta0 = ds[0]
    print(f"{model_name} DELTA_RESIDUE_PLUS_WILD: input_dim={x0.shape[0]}, label={y0:.4f}, region={meta0['region']}")

Expected: ESM-2=3840, AbLang2=1440. Same as Exp 3 in NB05_oscar.ipynb.

Record confirmed output here after running.

## Lambda Sweep

Runs all 8 combinations (4 lambdas x 2 models) sequentially. Each run is a separate
W&B entry. Results are collected in `sweep_results` keyed by `(model_name, lambda_cdr)`.

Lambda=0 must reproduce the Exp 3 unconstrained baseline (val Spearman ~0.700 ESM-2,
~0.662 AbLang2). If lambda=0 deviates substantially, check random seed and dataset
construction before proceeding.

**Constraint reminder:**
- Fires when model predicts |FR effect| > |CDR effect| in a batch
- Zero gradient when model already respects the CDR prior
- HER2 contributes zero constraint gradient (100% CDR H3, no FR)
- Lysozyme is the primary gradient source (66% FR, 1390 FR mutations)

In [ ]:
LAMBDAS = [0.0, 0.1, 0.5, 1.0]
MODELS  = ['esm2', 'ablang2']
INPUT_DIMS = {'esm2': 3840, 'ablang2': 1440}

sweep_results = {}

for model_name in MODELS:
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_RESIDUE_PLUS_WILD,
        model_name=model_name,
    )

    train_ds = Subset(ds_full, train_idx)
    val_ds   = Subset(ds_full, val_idx)
    test_ds  = Subset(ds_full, test_idx)
    input_dim = INPUT_DIMS[model_name]

    for lam in LAMBDAS:
        print(f"\n--- {model_name} | lambda={lam} ---")

        cfg = TrainConfig(
            model_name=model_name,
            embedding_strategy='delta_residue_plus_wild',
            lr=1e-3,
            epochs=100,
            batch_size=64,
            hidden_dims=[256, 128],
            dropout=0.1,
            lambda_cdr=lam,
            seed=42,
            patience=10,
            wandb_run_name=f"{model_name}_exp3_lambda{lam}",
        )

        result = train_abagym(cfg, train_ds, val_ds, input_dim, DEVICE)
        result['test_ds'] = test_ds
        sweep_results[(model_name, lam)] = result

        print(f"  Best epoch: {result['best_epoch']}")
        print(f"  Best val Spearman (all): {result['best_val_spearman']:.4f}")

Record confirmed val Spearman for each (model, lambda) combination here after running.

Key check: lambda=0 should match Exp 3 baseline (ESM-2 ~0.700, AbLang2 ~0.662).
If it does not, flag before proceeding to test evaluation.

## Test Evaluation

In [ ]:
print("=== Experiment 6: CDR Constraint Lambda Sweep -- Test Results ===")
for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Model: {model_name.upper()}")
    print(f"{'='*60}")
    for lam in LAMBDAS:
        result = sweep_results[(model_name, lam)]
        metrics = evaluate_abagym(result['model'], result['test_ds'], DEVICE)
        print(f"\n  lambda={lam}")
        print(f"    Aggregate Spearman (all 5):     {metrics['aggregate']:.4f}")
        print(f"    Aggregate Spearman (excl HER2): {metrics['exclude_her2']:.4f}")
        print(f"    HER2 Spearman:                  {metrics['HER2']:.4f}")
        print(f"    Per-dataset:")
        for ds, r in sorted(metrics['per_dataset'].items()):
            print(f"      {ds:<35} {r:.4f}")
        sweep_results[(model_name, lam)]['test_metrics'] = metrics

Record confirmed test results for all 8 runs here after running.

Key questions:
- Does any lambda improve over the unconstrained baseline (lambda=0) for ESM-2?
- Does any lambda improve over the unconstrained baseline for AbLang2?
- Is the optimal lambda different between models?
- Does the constraint help or hurt HER2 (all CDR H3, zero constraint gradient)?

## Summary Table

In [ ]:
rows = []
for model_name in MODELS:
    for lam in LAMBDAS:
        m = sweep_results[(model_name, lam)]['test_metrics']
        row = {
            'Model': model_name,
            'Lambda': lam,
            'Spearman_all': round(m['aggregate'], 4),
            'Spearman_excl_HER2': round(m['exclude_her2'], 4),
            'HER2': round(m['HER2'], 4),
        }
        for ds, r in m['per_dataset'].items():
            row[ds] = round(r, 4)
        rows.append(row)

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

Record confirmed summary table here after running. This is the primary
deliverable for Experiment 6 and feeds directly into the final results notebook.

## Interpretation

Record findings and interpretation here after results are confirmed.

Key questions to address:
1. Does the CDR constraint improve performance for either model?
2. Is the effect different between ESM-2 and AbLang2, and does the direction match predictions?
3. What does the result say about the neurosymbolic hypothesis?

**Prior predictions from EDA geometry:**
- ESM-2: inverse CDR prior at both levels (FR > CDR). Constraint directly opposes
  ESM-2's learned geometry. Could help by correcting the bias, or hurt by fighting
  the representation.
- AbLang2: inverse prior at sequence level but correct prior at residue level (CDR > FR).
  For Exp 3 (which includes residue-level delta), the constraint may fire less and
  have smaller impact on AbLang2.